# Isomap: Isometric Mapping

## Co je Isomap?

Isomap (Isometric Mapping) je nelineární metoda redukce dimenzionality, která zachovává geodetické vzdálenosti mezi body. Na rozdíl od lineárních metod jako PCA, Isomap dokáže zachytit nelineární strukturu dat a je obzvláště efektivní pro data ležící na tzv. "variety" (manifold).

### Klíčové vlastnosti Isomap:

- **Zachování geodetických vzdáleností**: Místo Euklidovské vzdálenosti používá vzdálenosti podél variety
- **Nelineární redukce**: Zachycuje nelineární struktury v datech
- **Založená na grafech**: Využívá graf nejbližších sousedů k aproximaci variety
- **Globální optimalizace**: Na rozdíl od LLE zachovává globální strukturu dat

### Kdy použít Isomap:

1. **Nelineární data**: Když vaše data leží na nebo blízko nelineární variety
2. **Vizualizace vysokodimenzionálních dat**: Pro vizualizaci složitých dat ve 2D nebo 3D
3. **Zachování vzdáleností**: Když je důležité zachovat vzdálenosti mezi body
4. **Předzpracování**: Jako krok předzpracování před použitím jiných algoritmů strojového učení

### Jak Isomap funguje:

1. Konstrukce grafu sousedství: Pro každý bod najde k nejbližších sousedů
2. Výpočet nejkratších cest: Spočítá geodetické vzdálenosti (nejkratší cesty v grafu) mezi všemi body
3. Aplikace MDS: Použije Multi-Dimensional Scaling (MDS) na matici geodetických vzdáleností

V scikit-learn je Isomap implementován ve třídě `Isomap` v modulu `sklearn.manifold`.

In [ ]:
# Import potřebných knihoven
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import Isomap, TSNE, LocallyLinearEmbedding, SpectralEmbedding
from sklearn.decomposition import PCA, KernelPCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_swiss_roll, make_s_curve, fetch_olivetti_faces, load_digits, load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import kneighbors_graph
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import time
from mpl_toolkits.mplot3d import Axes3D  # Pro 3D grafy
import warnings

# Pro lepší vizualizaci grafů
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("viridis")
warnings.filterwarnings('ignore')
np.random.seed(42)

## 1. Základní použití Isomap

Nejprve si ukážeme základní použití Isomap na syntetickém datasetu, který má zřejmou nelineární strukturu - švýcarský závitek (Swiss Roll).

In [ ]:
# Vytvoření Swiss Roll datasetu
n_samples = 1000
noise = 0.05
X_swiss_roll, color_swiss_roll = make_swiss_roll(n_samples=n_samples, noise=noise, random_state=42)

# Vizualizace původních 3D dat
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(X_swiss_roll[:, 0], X_swiss_roll[:, 1], X_swiss_roll[:, 2], 
          c=color_swiss_roll, cmap='viridis', s=30, alpha=0.8)
ax.set_title('Swiss Roll dataset', fontsize=16)
ax.view_init(elev=10, azim=70)  # Nastavení úhlu pohledu
plt.tight_layout()
plt.show()

In [ ]:
# Aplikace Isomap
start_time = time.time()
isomap = Isomap(n_components=2, n_neighbors=10)
X_isomap = isomap.fit_transform(X_swiss_roll)
isomap_time = time.time() - start_time

print(f"Doba výpočtu Isomap: {isomap_time:.4f} sekund")

# Vizualizace výsledků Isomap
plt.figure(figsize=(10, 8))
plt.scatter(X_isomap[:, 0], X_isomap[:, 1], c=color_swiss_roll, cmap='viridis', s=30, alpha=0.8)
plt.colorbar(label='Pozice na původním závitku')
plt.title('Swiss Roll data po aplikaci Isomap', fontsize=16)
plt.xlabel('Komponenta 1', fontsize=12)
plt.ylabel('Komponenta 2', fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Porovnání s jinými metodami redukce dimenzionality
# 1. PCA (lineární metoda)
start_time = time.time()
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_swiss_roll)
pca_time = time.time() - start_time

# 2. t-SNE (nelineární metoda)
start_time = time.time()
tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X_swiss_roll)
tsne_time = time.time() - start_time

# 3. LLE (Locally Linear Embedding)
start_time = time.time()
lle = LocallyLinearEmbedding(n_components=2, n_neighbors=10, random_state=42)
X_lle = lle.fit_transform(X_swiss_roll)
lle_time = time.time() - start_time

# Vizualizace výsledků všech metod
plt.figure(figsize=(16, 12))

plt.subplot(2, 2, 1)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=color_swiss_roll, cmap='viridis', s=30, alpha=0.8)
plt.title(f'PCA ({pca_time:.2f}s)', fontsize=14)
plt.colorbar(label='Pozice')
plt.grid(True)

plt.subplot(2, 2, 2)
plt.scatter(X_isomap[:, 0], X_isomap[:, 1], c=color_swiss_roll, cmap='viridis', s=30, alpha=0.8)
plt.title(f'Isomap ({isomap_time:.2f}s)', fontsize=14)
plt.colorbar(label='Pozice')
plt.grid(True)

plt.subplot(2, 2, 3)
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=color_swiss_roll, cmap='viridis', s=30, alpha=0.8)
plt.title(f't-SNE ({tsne_time:.2f}s)', fontsize=14)
plt.colorbar(label='Pozice')
plt.grid(True)

plt.subplot(2, 2, 4)
plt.scatter(X_lle[:, 0], X_lle[:, 1], c=color_swiss_roll, cmap='viridis', s=30, alpha=0.8)
plt.title(f'LLE ({lle_time:.2f}s)', fontsize=14)
plt.colorbar(label='Pozice')
plt.grid(True)

plt.tight_layout()
plt.suptitle('Porovnání metod redukce dimenzionality na Swiss Roll datasetu', fontsize=16, y=1.02)
plt.show()

# Shrnutí času výpočtu
print(f"PCA: {pca_time:.4f} sekund")
print(f"Isomap: {isomap_time:.4f} sekund")
print(f"t-SNE: {tsne_time:.4f} sekund")
print(f"LLE: {lle_time:.4f} sekund")

## 2. Vliv parametru n_neighbors

Jeden z klíčových parametrů Isomap je `n_neighbors`, který určuje, kolik nejbližších sousedů se použije k vytvoření grafu. Podívejme se, jak tento parametr ovlivňuje výsledky.

In [ ]:
# Vytvoříme nová syntetická data - S-křivku
X_s_curve, color_s_curve = make_s_curve(n_samples=1000, noise=0.05, random_state=42)

# Vizualizace původních 3D dat
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(X_s_curve[:, 0], X_s_curve[:, 1], X_s_curve[:, 2], 
          c=color_s_curve, cmap='viridis', s=30, alpha=0.8)
ax.set_title('S-křivka dataset', fontsize=16)
ax.view_init(elev=10, azim=70)  # Nastavení úhlu pohledu
plt.tight_layout()
plt.show()

# Testování různých hodnot n_neighbors
neighbors = [5, 10, 20, 50, 100]
results = {}

for n in neighbors:
    start_time = time.time()
    isomap = Isomap(n_components=2, n_neighbors=n)
    results[n] = {
        'embedding': isomap.fit_transform(X_s_curve),
        'time': time.time() - start_time
    }
    print(f"n_neighbors={n}: {results[n]['time']:.4f} sekund")

# Vizualizace výsledků pro různé hodnoty n_neighbors
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

# Původní data ve 2D projekci
axes[0].scatter(X_s_curve[:, 0], X_s_curve[:, 2], c=color_s_curve, cmap='viridis', s=30, alpha=0.8)
axes[0].set_title('Původní data (2D projekce)', fontsize=14)
axes[0].grid(True)

# Výsledky pro různé n_neighbors
for i, n in enumerate(neighbors):
    axes[i+1].scatter(results[n]['embedding'][:, 0], results[n]['embedding'][:, 1], 
                    c=color_s_curve, cmap='viridis', s=30, alpha=0.8)
    axes[i+1].set_title(f'n_neighbors={n} ({results[n]["time"]:.2f}s)', fontsize=14)
    axes[i+1].grid(True)

plt.tight_layout()
plt.show()

## 3. Aplikace Isomap na reálná data - MNIST Digits

Nyní aplikujeme Isomap na reálný dataset - MNIST ručně psané číslice pro vizualizaci vysoko-dimenzionálních dat.

In [ ]:
# Načtení MNIST digits datasetu
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

print(f"Tvar datasetu: {X_digits.shape}")
print(f"Počet unikátních tříd: {len(np.unique(y_digits))}")

# Zobrazení několika příkladů číslic
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(digits.images[i], cmap='gray')
    ax.set_title(f'Číslice: {y_digits[i]}')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Standardizace dat
scaler = StandardScaler()
X_digits_scaled = scaler.fit_transform(X_digits)

# Aplikace Isomap na MNIST
start_time = time.time()
isomap_digits = Isomap(n_components=2, n_neighbors=10)
X_digits_isomap = isomap_digits.fit_transform(X_digits_scaled)
isomap_digits_time = time.time() - start_time
print(f"Doba výpočtu Isomap pro MNIST: {isomap_digits_time:.4f} sekund")

# Pro porovnání PCA
start_time = time.time()
pca_digits = PCA(n_components=2, random_state=42)
X_digits_pca = pca_digits.fit_transform(X_digits_scaled)
pca_digits_time = time.time() - start_time
print(f"Doba výpočtu PCA pro MNIST: {pca_digits_time:.4f} sekund")

In [ ]:
# Vizualizace Isomap a PCA výsledků s barevným odlišením číslic
plt.figure(figsize=(16, 7))

plt.subplot(1, 2, 1)
scatter = plt.scatter(X_digits_isomap[:, 0], X_digits_isomap[:, 1], 
                     c=y_digits, cmap='tab10', s=30, alpha=0.8)
plt.colorbar(label='Číslice')
plt.title(f'MNIST: Isomap ({isomap_digits_time:.2f}s)', fontsize=14)
plt.xlabel('Komponenta 1', fontsize=12)
plt.ylabel('Komponenta 2', fontsize=12)
plt.grid(True)

plt.subplot(1, 2, 2)
plt.scatter(X_digits_pca[:, 0], X_digits_pca[:, 1], 
          c=y_digits, cmap='tab10', s=30, alpha=0.8)
plt.colorbar(label='Číslice')
plt.title(f'MNIST: PCA ({pca_digits_time:.2f}s)', fontsize=14)
plt.xlabel('Komponenta 1', fontsize=12)
plt.ylabel('Komponenta 2', fontsize=12)
plt.grid(True)

plt.tight_layout()
plt.show()

# Zobrazíme vybrané číslice přímo v projekci
def plot_digits_in_embedding(X_embedded, images, targets, nrows=10, ncols=10, title=""):
    fig, ax = plt.subplots(figsize=(12, 10))
    ax.scatter(X_embedded[:, 0], X_embedded[:, 1], c=targets, cmap='tab10', alpha=0.1)
    
    # Vybereme náhodně některé body pro zobrazení
    indices = np.random.choice(range(len(X_embedded)), nrows * ncols, replace=False)
    
    for i, idx in enumerate(indices):
        # Pozice v embeddigu
        x, y = X_embedded[idx, 0], X_embedded[idx, 1]
        
        # Vložení obrázku
        img = images[idx].reshape(8, 8)
        img = (img - img.min()) / (img.max() - img.min())  # Normalizace pro zobrazení
        
        # Velikost obrázku v grafu
        img_size = 0.03
        
        # Přidání obrázku na pozici v embeddigu
        ax.imshow(img, extent=(x-img_size, x+img_size, y-img_size, y+img_size), 
                  cmap='gray', interpolation='nearest', zorder=2)
    
    ax.set_title(title, fontsize=14)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# Zobrazení číslic v Isomap projekci
plot_digits_in_embedding(X_digits_isomap, digits.images, y_digits, title="Číslice v Isomap projekci")

## 4. Zachování vzdáleností

Jednou z hlavních výhod Isomap je zachování geodetických vzdáleností. Podívejme se, jak dobře jsou tyto vzdálenosti zachovány ve srovnání s jinými metodami.

In [ ]:
# Vytvoření Swiss Roll datasetu s menším počtem bodů pro rychlejší výpočet
n_samples_small = 300
X_swiss_small, color_swiss_small = make_swiss_roll(n_samples=n_samples_small, noise=0.05, random_state=42)

# Výpočet geodetických vzdáleností v původních datech
# 1. Vytvoření grafu sousedství
n_neighbors_graph = 10
kng = kneighbors_graph(X_swiss_small, n_neighbors_graph, mode='distance', include_self=False)
kng_dense = kng.toarray()

# 2. Nastavení nekonečné vzdálenosti pro nepropojené body
kng_dense[kng_dense == 0] = np.inf
np.fill_diagonal(kng_dense, 0)

# 3. Výpočet nejkratších cest pomocí Floydova-Warshallova algoritmu
def floyd_warshall(graph):
    n = graph.shape[0]
    dist = graph.copy()
    
    for k in range(n):
        for i in range(n):
            for j in range(n):
                if dist[i, k] + dist[k, j] < dist[i, j]:
                    dist[i, j] = dist[i, k] + dist[k, j]
    
    return dist

try:
    # Floydův-Warshallův algoritmus může být pomalý pro velké grafy
    geodesic_distances = floyd_warshall(kng_dense)
    
    # Nahrazení nekonečných vzdáleností maximální konečnou vzdáleností
    max_dist = np.max(geodesic_distances[geodesic_distances != np.inf])
    geodesic_distances[np.isinf(geodesic_distances)] = max_dist * 2
    
    # Aplikace redukce dimenzionality
    isomap_small = Isomap(n_components=2, n_neighbors=n_neighbors_graph)
    X_isomap_small = isomap_small.fit_transform(X_swiss_small)
    
    pca_small = PCA(n_components=2, random_state=42)
    X_pca_small = pca_small.fit_transform(X_swiss_small)
    
    # Výpočet euklidovských vzdáleností v redukovaných datech
    isomap_distances = pairwise_distances(X_isomap_small)
    pca_distances = pairwise_distances(X_pca_small)
    
    # Normalizace vzdáleností pro lepší porovnání
    geodesic_distances /= np.max(geodesic_distances)
    isomap_distances /= np.max(isomap_distances)
    pca_distances /= np.max(pca_distances)
    
    # Vybereme náhodné páry bodů pro porovnání
    np.random.seed(42)
    n_pairs = 100
    indices = np.random.choice(range(n_samples_small), size=(n_pairs, 2), replace=False)
    
    # Extrakce vzdáleností pro vybrané páry bodů
    geodesic_pairs = np.array([geodesic_distances[i, j] for i, j in indices])
    isomap_pairs = np.array([isomap_distances[i, j] for i, j in indices])
    pca_pairs = np.array([pca_distances[i, j] for i, j in indices])
    
    # Vizualizace porovnání zachování vzdáleností
    plt.figure(figsize=(14, 6))
    
    plt.subplot(1, 2, 1)
    plt.scatter(geodesic_pairs, isomap_pairs, alpha=0.7)
    plt.plot([0, 1], [0, 1], 'r--')  # Přímka y=x pro perfektní zachování
    plt.title('Isomap: Zachování geodetických vzdáleností', fontsize=14)
    plt.xlabel('Normalizované geodetické vzdálenosti', fontsize=12)
    plt.ylabel('Normalizované Isomap vzdálenosti', fontsize=12)
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.scatter(geodesic_pairs, pca_pairs, alpha=0.7)
    plt.plot([0, 1], [0, 1], 'r--')  # Přímka y=x pro perfektní zachování
    plt.title('PCA: Zachování geodetických vzdáleností', fontsize=14)
    plt.xlabel('Normalizované geodetické vzdálenosti', fontsize=12)
    plt.ylabel('Normalizované PCA vzdálenosti', fontsize=12)
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Výpočet korelace mezi původními a redukovanými vzdálenostmi
    isomap_corr = np.corrcoef(geodesic_pairs, isomap_pairs)[0, 1]
    pca_corr = np.corrcoef(geodesic_pairs, pca_pairs)[0, 1]
    
    print(f"Korelace mezi geodetickými a Isomap vzdálenostmi: {isomap_corr:.4f}")
    print(f"Korelace mezi geodetickými a PCA vzdálenostmi: {pca_corr:.4f}")
except Exception as e:
    print("Chyba při výpočtu geodetických vzdáleností:", e)

## 5. Využití Isomap pro klasifikaci

Podívejme se, jak dobře funguje Isomap jako předzpracování pro klasifikační úlohy.

In [ ]:
# Načtení datasetu Iris
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

print(f"Iris dataset: {X_iris.shape}")
print(f"Cílové třídy: {np.unique(y_iris)}")

# Rozdělení na trénovací a testovací sadu
X_train, X_test, y_train, y_test = train_test_split(X_iris, y_iris, test_size=0.3, random_state=42)

# Porovnání různých metod redukce dimenzionality
methods = {
    'Původní data': None,
    'PCA': PCA(n_components=2, random_state=42),
    'Isomap': Isomap(n_components=2, n_neighbors=5),
    't-SNE': TSNE(n_components=2, random_state=42),
    'LLE': LocallyLinearEmbedding(n_components=2, n_neighbors=5, random_state=42)
}

results = []

# Standardizace dat
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

for method_name, reducer in methods.items():
    if reducer is None:
        # Použití původních dat
        X_train_reduced = X_train_scaled
        X_test_reduced = X_test_scaled
    else:
        # Redukce dimenzionality
        if method_name == 't-SNE':
            # t-SNE nemá metodu transform, proto je třeba trénovat znovu na test datech
            X_train_reduced = reducer.fit_transform(X_train_scaled)
            X_test_reduced = TSNE(n_components=2, random_state=42).fit_transform(X_test_scaled)
        else:
            X_train_reduced = reducer.fit_transform(X_train_scaled)
            X_test_reduced = reducer.transform(X_test_scaled)
    
    # Klasifikace s RandomForest
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train_reduced, y_train)
    y_pred = clf.predict(X_test_reduced)
    
    # Vyhodnocení přesnosti
    accuracy = accuracy_score(y_test, y_pred)
    
    results.append({
        'Metoda': method_name,
        'Dimenze': X_train_reduced.shape[1],
        'Přesnost': accuracy
    })
    
    print(f"{method_name}: {accuracy:.4f}")

# Vizualizace výsledků
results_df = pd.DataFrame(results)
plt.figure(figsize=(10, 6))
sns.barplot(x='Metoda', y='Přesnost', data=results_df)
plt.title('Přesnost klasifikace po redukci dimenzionality', fontsize=14)
plt.ylim(0, 1.0)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Analýza časové a paměťové složitosti Isomap

Zkoumejme, jak se mění výpočetní čas a paměťové nároky Isomap s rostoucím počtem vzorků a dimenzí.

In [ ]:
# Test škálování s počtem vzorků
sample_sizes = [100, 500, 1000, 2000, 5000]
dim = 20  # Pevná dimenzionalita

times_isomap = []
times_pca = []

for n in sample_sizes:
    # Generování náhodných dat
    np.random.seed(42)
    X = np.random.randn(n, dim)
    
    # Měření času pro Isomap
    start_time = time.time()
    try:
        isomap = Isomap(n_components=2, n_neighbors=10)
        isomap.fit_transform(X)
        isomap_time = time.time() - start_time
    except MemoryError:
        isomap_time = float('nan')  # V případě nedostatku paměti
    times_isomap.append(isomap_time)
    
    # Měření času pro PCA jako benchmark
    start_time = time.time()
    pca = PCA(n_components=2)
    pca.fit_transform(X)
    pca_time = time.time() - start_time
    times_pca.append(pca_time)
    
    print(f"n={n}, Isomap: {isomap_time:.4f}s, PCA: {pca_time:.4f}s")

# Vizualizace výsledků
plt.figure(figsize=(12, 6))
plt.plot(sample_sizes, times_isomap, 'o-', linewidth=2, label='Isomap')
plt.plot(sample_sizes, times_pca, 'x-', linewidth=2, label='PCA')
plt.xlabel('Počet vzorků', fontsize=12)
plt.ylabel('Čas [s]', fontsize=12)
plt.title('Škálování výpočetního času s počtem vzorků', fontsize=14)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Test škálování s dimenzionalitou
dimensions = [10, 50, 100, 200, 500]
n_samples = 500  # Pevný počet vzorků

times_isomap_dim = []
times_pca_dim = []

for d in dimensions:
    # Generování náhodných dat
    np.random.seed(42)
    X = np.random.randn(n_samples, d)
    
    # Měření času pro Isomap
    start_time = time.time()
    isomap = Isomap(n_components=2, n_neighbors=10)
    isomap.fit_transform(X)
    isomap_time = time.time() - start_time
    times_isomap_dim.append(isomap_time)
    
    # Měření času pro PCA jako benchmark
    start_time = time.time()
    pca = PCA(n_components=2)
    pca.fit_transform(X)
    pca_time = time.time() - start_time
    times_pca_dim.append(pca_time)
    
    print(f"d={d}, Isomap: {isomap_time:.4f}s, PCA: {pca_time:.4f}s")

# Vizualizace výsledků
plt.figure(figsize=(12, 6))
plt.plot(dimensions, times_isomap_dim, 'o-', linewidth=2, label='Isomap')
plt.plot(dimensions, times_pca_dim, 'x-', linewidth=2, label='PCA')
plt.xlabel('Dimenzionalita', fontsize=12)
plt.ylabel('Čas [s]', fontsize=12)
plt.title('Škálování výpočetního času s dimenzionalitou', fontsize=14)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## 7. Shrnutí a doporučení pro použití Isomap

### Výhody Isomap:

1. **Zachování globální struktury** - Isomap dokáže zachovat geodetické vzdálenosti mezi body, což vede k intuitivní reprezentaci dat.
2. **Nelineární redukce dimenzionality** - Efektivně zvládá data ležící na nelineárních varietách, kde lineární metody selhávají.
3. **Teoretické základy** - Založeno na solidních matematických základech spektrální teorie grafů.
4. **Interpretovatelné výsledky** - Výsledná projekce často odpovídá intuitivnímu "rozbalení" variety.
5. **Flexibilita** - Parametr `n_neighbors` umožňuje přizpůsobit granularitu analýzy dat.

### Nevýhody Isomap:

1. **Výpočetní náročnost** - Časová složitost O(N³) a paměťová složitost O(N²), kde N je počet vzorků, což limituje použití na větších datasetech.
2. **Citlivost na parametr n_neighbors** - Příliš malá nebo velká hodnota může vést k neoptimálním výsledkům.
3. **Problémy s "děravými" daty** - Pokud data obsahují "díry" nebo nejsou spojitá, Isomap může selhávat.
4. **Citlivost na šum** - Šum může výrazně ovlivnit konstrukci grafu a následné geodetické vzdálenosti.
5. **Obtížné out-of-sample rozšíření** - Aplikace na nová data může být komplikovaná a vyžaduje dodatečné výpočty.

### Doporučení pro použití:

- **Kdy použít Isomap**:
  - Pro vizualizaci dat s nelineární strukturou
  - Když je důležité zachovat geodetické vzdálenosti mezi body
  - Pro data, která leží na spojité varietě (bez děr a přerušení)
  - Jako předzpracování před klasifikací nebo shlukováním, když jsou důležité globální vztahy

- **Alternativy k zvážení**:
  - Pro velmi velké datové sady: t-SNE, UMAP nebo SpectralEmbedding
  - Pro rychlou explorační analýzu: PCA
  - Pro zachování lokální struktury: LLE, Laplacian Eigenmaps
  - Pro lepší škálovatelnost: kernelPCA s aproximačními metodami

- **Optimální nastavení parametrů**:
  - `n_neighbors`: Obvykle mezi 5-20 pro menší datasety, případně větší pro větší datasety (vždy experimentujte s několika hodnotami)
  - `n_components`: Pro vizualizaci 2-3, pro předzpracování zkuste více hodnot a vyberte optimální podle následného úkolu
  - `path_method`: Pro větší datasety zvažte 'auto' nebo 'FW' (Floyd-Warshall)
  - `neighbors_algorithm`: Pro vyšší dimenze použijte 'auto' nebo 'brute'

### Závěr:

Isomap je mocná technika nelineární redukce dimenzionality, která je obzvláště účinná při zachování globální struktury dat ležících na varietě. Její hlavní nevýhodou je škálovatelnost s počtem vzorků, která může být omezující pro velké datové sady. Přesto, pro menší až středně velké datasety s komplexní nelineární strukturou, poskytuje Isomap intuitivní a užitečné výsledky, které mohou vést k lepšímu pochopení dat a vylepšení následných algoritmů strojového učení.